# Phase-3D campaign progress

Ops status over `results/phase3d/hy{hy}/{electric_hx{hx}|magnetic_hz{hz}}/L{L}/` --
coverage of the planned grid, health of what has landed, and a running read of the
transition location per (cut, L). Everything below degrades to an empty table/plot on
a partial or empty campaign -- rerun any time, it never assumes the run is finished.

The interactive drill-down view (per-point learning curves, order-parameter sweeps) is
a separate published Artifact fed by `phase3d_status.export_viewer` -- not duplicated
here. Locator fits reuse `analysis/scripts/transition_fit.py` (untracked peer module).


## 1. Config

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display

# repo root, found from wherever nbconvert's cwd lands (project convention: notebooks
# run with cwd = analysis/notebooks/, but this works regardless)
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "analysis" / "scripts"))
import phase3d_status as ps           # noqa: E402
import transition_fit as tf           # noqa: E402  (untracked peer module; import-only)

# --- knobs --------------------------------------------------------------------------
# Env-var overridable so the SAME committed config runs on today's empty results/phase3d
# and on a stand-in tree (set PHASE3D_CAMPAIGN / PHASE3D_DATA_CURVES for a dry run).
CAMPAIGN = Path(os.environ.get("PHASE3D_CAMPAIGN", ROOT / "results" / "phase3d"))
DATA_CURVES = Path(os.environ.get("PHASE3D_DATA_CURVES", ROOT / "data" / "tc_nqs" / "phase3d"))
FIGS = ROOT / "analysis" / "figs"
# DATA_CURVES isn't read below (the ops sections don't plot curves) -- it's the
# curves_root the drill-down-viewer JSON export (phase3d_status.export_viewer /
# --export-viewer) expects, kept here so both live off the same knob.
HY_LIST = None        # None = auto-detect from what's landed/planned; else e.g. [0.0, 0.2, 0.4]
L_LIST = None         # None = auto-detect; else e.g. [4, 5, 6]
MIN_POINTS = 5        # locator gate: >= this many landed points on a cut before fitting h_c(L)
WRITE_STATUS = True   # section 5: write CAMPAIGN/STATUS.md

print(f"ROOT        = {ROOT}")
print(f"CAMPAIGN    = {CAMPAIGN}  (exists: {CAMPAIGN.exists()})")
print(f"DATA_CURVES = {DATA_CURVES}  (exists: {DATA_CURVES.exists()})")


In [ ]:
def show_scrollable(df, max_height=360):
    """House style: wide tables get their own scrollable container."""
    if df is None or df.empty:
        display(HTML('<i>(no rows)</i>'))
        return
    html = df.to_html(index=False, na_rep="--", float_format=lambda x: f"{x:.4g}")
    display(HTML(f'<div style="max-width:100%; overflow-x:auto; max-height:{max_height}px; '
                 f'overflow-y:auto; border:1px solid #ddd; padding:4px;">{html}</div>'))


# --- load everything once ------------------------------------------------------------
finals = ps.add_health(ps.load_finals(CAMPAIGN))
manifests = ps.load_manifests(CAMPAIGN)
watch = ps.load_watch_state(CAMPAIGN)
planned = ps.load_planned(CAMPAIGN)
coverage = ps.coverage(planned, finals)
locators = ps.partial_locators(CAMPAIGN, finals, min_points=MIN_POINTS)

if HY_LIST is None:
    HY_LIST = sorted((set(finals["hy"]) if len(finals) else set())
                      | (set(coverage["hy"]) if len(coverage) else set()))
if L_LIST is None:
    L_LIST = sorted((set(finals["L"]) if len(finals) else set())
                     | (set(coverage["L"]) if len(coverage) else set()))
CUTS = ["electric", "magnetic"]

print(f"{len(finals)} landed runs, {len(manifests)} manifest rows, {len(watch)} watch_state entries")
print(f"hy in {HY_LIST}")
print(f"L  in {L_LIST}")


## 2. Coverage heat-grid
Rows = cuts, columns = L, one panel per h_y. Cell text is landed/planned; colour is the
landed fraction; a red hatch marks a cell with at least one diverged run.


In [ ]:
def plot_coverage(hy_list, l_list, cov):
    if not hy_list or not l_list:
        print("No coverage data yet (empty campaign).")
        return
    fig, axes = plt.subplots(1, len(hy_list), figsize=(3.4 * len(hy_list), 2.8), squeeze=False)
    axes = axes[0]
    for ax, hy in zip(axes, hy_list):
        sub = cov[np.isclose(cov["hy"], hy)]
        grid = np.full((len(CUTS), len(l_list)), np.nan)
        for i, cut in enumerate(CUTS):
            for j, L in enumerate(l_list):
                row = sub[(sub.cut == cut) & (sub.L == L)]
                if len(row):
                    grid[i, j] = row.iloc[0]["frac"]
        ax.imshow(grid, vmin=0, vmax=1, cmap="plasma", aspect="auto")
        for i, cut in enumerate(CUTS):
            for j, L in enumerate(l_list):
                row = sub[(sub.cut == cut) & (sub.L == L)]
                if len(row):
                    r = row.iloc[0]
                    ax.text(j, i, f"{int(r['landed'])}/{int(r['planned'])}", ha="center", va="center",
                            fontsize=8, color="white" if r["frac"] > 0.5 else "black")
                    if r["diverged"] > 0:
                        ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False,
                                                    hatch="////", edgecolor="crimson", linewidth=0))
                else:
                    ax.text(j, i, "--", ha="center", va="center", fontsize=8, color="gray")
        ax.set_xticks(range(len(l_list)), [f"L={L}" for L in l_list])
        ax.set_yticks(range(len(CUTS)), CUTS)
        ax.set_title(f"h_y = {hy:g}")
        ax.spines[["top", "right", "left", "bottom"]].set_visible(False)
    fig.suptitle("Coverage: landed / planned  (hatched = has diverged runs)")
    fig.tight_layout()
    # plt.savefig(FIGS / "phase3d_coverage.png", dpi=300, bbox_inches="tight")
    plt.show()


plot_coverage(HY_LIST, L_LIST, coverage)


## 3. Job / health table
Every submitted job (manifest) joined with its live state (watch_state) and, once
landed, its final observables -- state, last step, diverged, warm-started, E0 vs the
h=0 bound, Vscore.


In [ ]:
job_tbl = ps.job_table(CAMPAIGN, df=finals, manifests=manifests, watch=watch)
if job_tbl.empty:
    print("No jobs recorded yet (no manifests or finals under CAMPAIGN).")
else:
    show_scrollable(job_tbl)


## 4. Partial locators
Electric cuts: per (h_y, L) with >= `MIN_POINTS` landed points, h_c(L) from
`transition_fit` (richards-inflection central value, the production marker policy).
Magnetic (chain) cuts: per branch, the last landed field and whether the up/dn energy
branches have crossed. Exact thermodynamic anchors and the peer's banked off-axis FSS
values (h_y=0, richards marker policy) are shown for reference -- they sit at different
(h_x, h_z) points than most campaign cuts, so agreement is not expected point-for-point.


In [ ]:
reference = pd.DataFrame([
    {"quantity": "hz_c(hx=0, hy=0)", "value": tf.EXACT["hz_c(hx=0,hy=0)"], "err": None,
     "source": "exact, (3+1)D Ising* duality (2nd order)"},
    {"quantity": "hx_c(hz=0, hy=0)", "value": tf.EXACT["hx_c(hz=0,hy=0)"], "err": None,
     "source": "exact, self-duality to 4D Wegner Z2 (1st order)"},
    {"quantity": "hz_c(hx=0.2, hy=0)", "value": 0.196, "err": 0.038,
     "source": "peer banked FSS, richards marker (off-axis electric cut)"},
    {"quantity": "hx_c(hz=0.1, hy=0)", "value": 0.98, "err": 0.19,
     "source": "peer banked FSS, richards marker (off-axis magnetic cut)"},
])
print("Reference anchors:")
show_scrollable(reference)

print("Electric-cut locators (partial FSS -- more L unlocks the true h_c(L->inf) fit):")
show_scrollable(locators["electric"])

print("Magnetic (chain) cut summary:")
show_scrollable(locators["chain"])


## 5. Write STATUS.md

In [ ]:
if WRITE_STATUS:
    out_path = CAMPAIGN / "STATUS.md"
    status_text = ps.write_status_md(CAMPAIGN, out_path, min_points=MIN_POINTS)
    print(f"wrote {out_path}")
    print(status_text)
else:
    print("WRITE_STATUS is False -- skipping STATUS.md.")
